# all-mpnet-base-v2 Sentence Embeddings - AWS Marketplace

Deploys **all-mpnet-base-v2 Sentence Embeddings** from AWS Marketplace as a SageMaker endpoint inside **your own AWS account**. Your data never leaves your VPC and there are no external API calls or token limits.

Sentence-transformer model producing 768-dimensional dense embeddings. Best-in-class for English semantic similarity tasks.

## Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # recommended real-time instance
ENDPOINT_NAME = "all-mpnet-base-v2-embeddings"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills per hour while it exists, so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Embed text

The endpoint accepts `application/json` shaped `{"sentences": ["...", "..."]}`.
Returns `{"embeddings": [[...]], "dim": 768}` — L2-normalised 768-dim vectors.

Vectors are L2-normalised; cosine similarity equals dot product. No query prefix needed.

In [ ]:
runtime = boto3.client("sagemaker-runtime")


def embed(texts):
    """Return embedding vectors for a list of texts (or a single string)."""
    if isinstance(texts, str):
        texts = [texts]
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"sentences": texts}),
    )
    return json.loads(response["Body"].read())["embeddings"]


vectors = embed("Machine learning is transforming enterprise AI.")
print("dimensions:", len(vectors[0]))
print("first 8 values:", [round(v, 5) for v in vectors[0][:8]])

# Semantic similarity: cosine similarity = dot product on L2-normalised vectors
texts = [
    "Machine learning is a subset of AI.",
    "Deep learning powers modern language models.",
    "The stock market fell sharply today.",
]
doc_vectors = embed(texts)
query_vec = embed("What is machine learning?")[0]

scored = [
    (sum(q * d for q, d in zip(query_vec, dv)), t)
    for dv, t in zip(doc_vectors, texts)
]
for score, text in sorted(scored, reverse=True):
    print(f"{score:.4f}  {text}")

## 4. Batch transform for offline workloads

For processing large datasets without a live endpoint, batch transform avoids paying for an always-on endpoint. Input is JSON Lines, one `{"inputs": "..."}` object per line.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type=INSTANCE_TYPE,
#     output_path=f"s3://{session.default_bucket()}/all-mpnet-base-v2-embeddings/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/all-mpnet-base-v2-embeddings-input/",
#     content_type="application/json",
# )
# transformer.wait()

## 5. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)